# AVL Tree: Build and Rotation Practice

## Problem

Given the insertion sequence **[30, 20, 10, 25, 40, 35, 50]**, for each step:

1. Draw the AVL tree after each insertion
2. Annotate each node's **balance factor**
3. If imbalance occurs, identify the **type** (LL/RR/LR/RL) and **rotation axis**
4. Draw the tree after rotation
5. Write the final **inorder traversal** to verify BST property

---

## AVL Tree Core Concepts

- **Balance Factor (BF)**: `bf = height(left) - height(right)`
- **AVL Condition**: |bf| <= 1 for every node
- **Four Rotation Types**:

| Type | Node BF | Child BF | Operation |
|------|:-------:|:--------:|-----------|
| **LL** | +2 | >= 0 | Right Rotate at node |
| **RR** | -2 | <= 0 | Left Rotate at node |
| **LR** | +2 | < 0 | Left(child) + Right(node) |
| **RL** | -2 | > 0 | Right(child) + Left(node) |

### Rotation Diagrams

```
LL (Right Rotate):       RR (Left Rotate):
    y (bf=+2)               x (bf=-2)
   / \                      / \
  x   T3                   T1  y
 / \                          / \
T1 T2                        T2 T3

LR (Left-Right):         RL (Right-Left):
    z (bf=+2)               z (bf=-2)
   / \                      / \
  x   T4                   T1  x
 / \                          / \
T1  y                        y  T4
   / \                      / \
  T2 T3                    T2 T3
```

**Key insight**: Rotation type is determined by the **child's BF** (not the inserted value):
- BF(node)=+2, BF(node.left)>=0 --> LL
- BF(node)=+2, BF(node.left)<0  --> LR
- BF(node)=-2, BF(node.right)<=0 --> RR
- BF(node)=-2, BF(node.right)>0  --> RL

---
## AVL Tree Implementation (with all 4 rotations)

In [1]:
class AVLNode:
    """AVL tree node"""
    def __init__(self, val: int):
        self.val = val
        self.left: AVLNode | None = None
        self.right: AVLNode | None = None
        self.height: int = 1


class AVLTree:
    """AVL self-balancing BST. Supports insert with LL/RR/LR/RL rotations."""

    def __init__(self):
        self.root: AVLNode | None = None
        self.last_rotation: str = ""

    # ============ Utilities ============

    @staticmethod
    def get_height(node: AVLNode | None) -> int:
        return node.height if node else 0

    @staticmethod
    def get_bf(node: AVLNode | None) -> int:
        """Balance factor = height(left) - height(right)"""
        if not node:
            return 0
        return AVLTree.get_height(node.left) - AVLTree.get_height(node.right)

    @staticmethod
    def update_height(node: AVLNode):
        node.height = 1 + max(
            AVLTree.get_height(node.left),
            AVLTree.get_height(node.right)
        )

    # ============ Rotations ============

    @staticmethod
    def rotate_right(y: AVLNode) -> AVLNode:
        """Right rotate (LL case). y is the imbalanced node."""
        x = y.left
        T2 = x.right
        x.right = y
        y.left = T2
        AVLTree.update_height(y)
        AVLTree.update_height(x)
        return x

    @staticmethod
    def rotate_left(x: AVLNode) -> AVLNode:
        """Left rotate (RR case). x is the imbalanced node."""
        y = x.right
        T2 = y.left
        y.left = x
        x.right = T2
        AVLTree.update_height(x)
        AVLTree.update_height(y)
        return y

    # ============ Insert ============

    def insert(self, val: int) -> None:
        self.last_rotation = ""
        self.root = self._insert(self.root, val)

    def _insert(self, node: AVLNode | None, val: int) -> AVLNode:
        # 1. Standard BST insert
        if not node:
            return AVLNode(val)
        if val < node.val:
            node.left = self._insert(node.left, val)
        elif val > node.val:
            node.right = self._insert(node.right, val)
        else:
            return node

        # 2. Update height
        self.update_height(node)

        # 3. Get balance factor
        bf = self.get_bf(node)

        # 4. Check 4 imbalance cases (using CHILD's BF, not inserted value)

        # LL: node.bf=+2, left child bf >= 0
        if bf > 1 and self.get_bf(node.left) >= 0:
            self.last_rotation = f"LL -> Right Rotate({node.val})"
            return self.rotate_right(node)

        # LR: node.bf=+2, left child bf < 0
        if bf > 1 and self.get_bf(node.left) < 0:
            self.last_rotation = f"LR -> Left({node.left.val}) + Right({node.val})"
            node.left = self.rotate_left(node.left)
            return self.rotate_right(node)

        # RR: node.bf=-2, right child bf <= 0
        if bf < -1 and self.get_bf(node.right) <= 0:
            self.last_rotation = f"RR -> Left Rotate({node.val})"
            return self.rotate_left(node)

        # RL: node.bf=-2, right child bf > 0
        if bf < -1 and self.get_bf(node.right) > 0:
            self.last_rotation = f"RL -> Right({node.right.val}) + Left({node.val})"
            node.right = self.rotate_right(node.right)
            return self.rotate_left(node)

        return node

    # ============ Inorder Traversal ============

    def inorder(self) -> list[int]:
        result = []
        self._inorder(self.root, result)
        return result

    def _inorder(self, node: AVLNode | None, result: list[int]):
        if node:
            self._inorder(node.left, result)
            result.append(node.val)
            self._inorder(node.right, result)

    # ============ Text Visualization ============

    def print_tree(self) -> None:
        if not self.root:
            print("(empty tree)")
            return
        self._print_node(self.root, "", True)

    def _print_node(self, node: AVLNode, prefix: str, is_tail: bool):
        if node.right:
            self._print_node(node.right, prefix + ("    " if is_tail else "|   "), False)
        connector = "+-- " if is_tail else "+-- "
        bf = self.get_bf(node)
        flag = " *** IMBALANCE!" if abs(bf) > 1 else ""
        print(prefix + connector + str(node.val) + f" (bf={bf:+d}){flag}")
        if node.left:
            self._print_node(node.left, prefix + ("|   " if is_tail else "    "), True)

    def print_bfs(self) -> None:
        if not self.root:
            return
        from collections import deque
        q = deque([(self.root, 0)])
        levels = {}
        while q:
            node, depth = q.popleft()
            levels.setdefault(depth, []).append(node)
            if node.left:
                q.append((node.left, depth + 1))
            if node.right:
                q.append((node.right, depth + 1))
        print("Level-order balance factors:")
        for depth in sorted(levels.keys()):
            info = ", ".join(f"{n.val}(bf={self.get_bf(n):+d})" for n in levels[depth])
            print(f"  L{depth}: {info}")


print("AVLTree class ready (LL/RR/LR/RL rotations).")
print("Starting step-by-step insertion demo...")

AVLTree class ready (LL/RR/LR/RL rotations).
Starting step-by-step insertion demo...


---
## Step-by-Step Insertion

Insert sequence: `[30, 20, 10, 25, 40, 35, 50]`

---
### Step 1: Insert 30

Empty tree. 30 becomes the root.

In [2]:
avl = AVLTree()
print(">>> insert(30)")
avl.insert(30)
avl.print_tree()
print()
avl.print_bfs()
print()
print("Status: root only, bf=0, balanced")

>>> insert(30)
+-- 30 (bf=+0)

Level-order balance factors:
  L0: 30(bf=+0)

Status: root only, bf=0, balanced


```
    30 (bf=0)
```

---
### Step 2: Insert 20

20 < 30, inserted as 30's left child.

In [3]:
print(">>> insert(20)")
avl.insert(20)
avl.print_tree()
print()
avl.print_bfs()
print()
print("Status: 30 bf=+1, within bounds, balanced")

>>> insert(20)
+-- 30 (bf=+1)
|   +-- 20 (bf=+0)

Level-order balance factors:
  L0: 30(bf=+1)
  L1: 20(bf=+0)

Status: 30 bf=+1, within bounds, balanced


```
    30 (bf=+1)
   /
 20 (bf=0)
```

---
### Step 3: Insert 10 -- LL Rotation!

10 < 30 -> left; 10 < 20 -> left. Forms a left-left chain.

**Before rotation:**
```
      30 (bf=+2) *** IMBALANCE
     /
   20 (bf=+1)
  /
10 (bf=0)
```

- **Imbalanced node**: 30, BF = +2
- **30's left child**: 20, BF = +1 (same sign, >=0)
- **Type**: **LL**
- **Rotation**: **Right Rotate at 30**

In [4]:
print(">>> insert(10)")
avl.insert(10)
print(f"Rotation: {avl.last_rotation}")
print()
avl.print_tree()
print()
avl.print_bfs()
print()
print("After LL rotation at 30, tree is balanced.")

>>> insert(10)
Rotation: LL -> Right Rotate(30)

    +-- 30 (bf=+0)
+-- 20 (bf=+0)
|   +-- 10 (bf=+0)

Level-order balance factors:
  L0: 20(bf=+0)
  L1: 10(bf=+0), 30(bf=+0)

After LL rotation at 30, tree is balanced.


**LL Rotation detail:**
```
Before:              After Right Rotate 30:
    30 (bf=+2)             20 (bf=0)
   /                      /  \
 20 (bf=+1)     -->     10    30
 /                      (bf=0)(bf=0)
10 (bf=0)
```

Tree height reduced from 3 to 2. All |bf| <= 1.

---
### Step 4: Insert 25

25 > 20 -> right; 25 < 30 -> left. Inserted as 30's left child.

In [5]:
print(">>> insert(25)")
avl.insert(25)
print(f"Rotation: {avl.last_rotation if avl.last_rotation else 'None'}")
print()
avl.print_tree()
print()
avl.print_bfs()
print()
print("Status: all |bf| <= 1, balanced")

>>> insert(25)
Rotation: None

    +-- 30 (bf=+1)
        +-- 25 (bf=+0)
+-- 20 (bf=-1)
|   +-- 10 (bf=+0)

Level-order balance factors:
  L0: 20(bf=-1)
  L1: 10(bf=+0), 30(bf=+1)
  L2: 25(bf=+0)

Status: all |bf| <= 1, balanced


```
      20 (bf=-1)
     /  \
 10(bf=0) 30(bf=+1)
          /
       25(bf=0)
```
All |bf| <= 1, balanced.

---
### Step 5: Insert 40

40 > 20 -> right; 40 > 30 -> right. Inserted as 30's right child.

In [6]:
print(">>> insert(40)")
avl.insert(40)
print(f"Rotation: {avl.last_rotation if avl.last_rotation else 'None'}")
print()
avl.print_tree()
print()
avl.print_bfs()
print()
print("Status: all |bf| <= 1, balanced")

>>> insert(40)
Rotation: None

    |   +-- 40 (bf=+0)
    +-- 30 (bf=+0)
        +-- 25 (bf=+0)
+-- 20 (bf=-1)
|   +-- 10 (bf=+0)

Level-order balance factors:
  L0: 20(bf=-1)
  L1: 10(bf=+0), 30(bf=+0)
  L2: 25(bf=+0), 40(bf=+0)

Status: all |bf| <= 1, balanced


```
      20 (bf=-1)
     /  \
 10(bf=0) 30(bf=0)
          /  \
      25(bf=0) 40(bf=0)
```
All |bf| <= 1, balanced. Note 30.bf went from +1 to 0 (balanced by the right child).

---
### Step 6: Insert 35 -- RR Rotation!

35 > 20 -> right; 35 > 30 -> right; 35 < 40 -> left. Inserted as 40's left child.

**Let's trace heights bottom-up after insertion:**

| Node | Left Height | Right Height | New Height | New BF |
|------|:-----------:|:------------:|:----------:|:------:|
| 35 | 0 | 0 | 1 | 0 |
| 40 | 1 (35) | 0 | 2 | **+1** |
| 30 | 1 (25) | 2 (40) | 3 | **-1** |
| 20 | 1 (10) | 3 (30) | 4 | **-2** ⚠️ |

Only **20** is imbalanced!

```
        20 (bf=-2) *** IMBALANCE
       /  \
   10(bf=0) 30(bf=-1)
            /  \
        25(bf=0) 40(bf=+1)
                /
             35(bf=0)
```

- **Imbalanced node**: 20, BF = -2
- **20's right child**: 30, BF = -1 (also negative, i.e. <= 0)
- Same sign -> **RR** case
- **Rotation**: **Left Rotate at 20**

**RR Rotation Details:**

Left rotate at 20 (pivot = 20.right = 30; T2 = 30.left = 25):
```
Before RR:                    After Left Rotate 20:
     20(bf=-2)                     30(bf=0)
    /  \                          /  \
  10   30(bf=-1)       -->     20(bf=0) 40(bf=+1)
      /  \                     / \      /
    25   40(bf=+1)          10   25   35
        /
      35
```

**Post-rotation verification:**
| Node | BF | Status |
|------|:--:|--------|
| 10, 25, 35 | 0 | Leaf |
| 20 | 0 | Left=10(h=1), Right=25(h=1) |
| 40 | +1 | Left=35(h=1), Right=None(h=0) |
| 30 | 0 | Left=20(h=2), Right=40(h=2) |

All |bf| <= 1 ✅ Only ONE rotation was needed.

> **Key insight**: Not every insertion causes cascading rotations. Here only 20 was imbalanced.
> We used the child's BF (not the inserted value) to determine the rotation type:
> `20.bf=-2` and `30.bf=-1 <= 0` -> RR.

In [7]:
print(">>> insert(35)")
print("=" * 50)
avl.insert(35)
print(f"Rotation: {avl.last_rotation}")
print("=" * 50)
print()
avl.print_tree()
print()
avl.print_bfs()
print()
print("Status: After RR rotation at 20, all |bf| <= 1, balanced")
print()
print("Only ONE node (20) was imbalanced with bf=-2.")
print("20.bf=-2, 30.bf=-1 (same sign) -> RR -> one left rotate fixed it.")

>>> insert(35)
Rotation: RR -> Left Rotate(20)

    +-- 40 (bf=+1)
        +-- 35 (bf=+0)
+-- 30 (bf=+0)
|       +-- 25 (bf=+0)
|   +-- 20 (bf=+0)
|   |   +-- 10 (bf=+0)

Level-order balance factors:
  L0: 30(bf=+0)
  L1: 20(bf=+0), 40(bf=+1)
  L2: 10(bf=+0), 25(bf=+0), 35(bf=+0)

Status: After RR rotation at 20, all |bf| <= 1, balanced

Only ONE node (20) was imbalanced with bf=-2.
20.bf=-2, 30.bf=-1 (same sign) -> RR -> one left rotate fixed it.


---
### Step 7: Insert 50 -- No Rotation Needed!

50 > 30 -> right; 50 > 40 -> right. Inserted as 40's right child.

**Before rotation:**
```
          30(bf=-1)
         /  \
     20(bf=0) 40(bf=+1)
     /  \     /
  10    25  35
  (bf=0)(bf=0)(bf=0)
```

After inserting 50 as 40's right child:

| Node | BF before | New BF | Status |
|------|:---------:|:------:|--------|
| 50 | - | 0 | New leaf |
| 40 | +1 | **0** | Now balanced by right child |
| 30 | -1 | **0** | Left=h=2, Right=h=2 |
| 20 | 0 | 0 | Unchanged |

**No node exceeds** |bf| <= 1! The tree is **perfectly balanced** after insertion.

```
          30(bf=0)      <- PERFECT!
         /  \
     20(bf=0) 40(bf=0)
     /  \     /  \
  10    25  35    50
 (bf=0)(bf=0)(bf=0)(bf=0)
```

> **Interesting observation**: Inserting 50 actually IMPROVED the tree's balance!
> 40 went from bf=+1 to bf=0 because the right child filled in the gap.
> This is a perfect AVL tree -- every node has bf=0.

In [8]:
print(">>> insert(50)")
avl.insert(50)
print(f"Rotation: {avl.last_rotation if avl.last_rotation else 'NONE (tree remained balanced)'}")
print()
avl.print_tree()
print()
avl.print_bfs()
print()
print("PERFECTLY balanced! Every node has bf=0.")
print("AVL tree height = 3 (optimal for 7 nodes).")

>>> insert(50)
Rotation: NONE (tree remained balanced)

    |   +-- 50 (bf=+0)
    +-- 40 (bf=+0)
        +-- 35 (bf=+0)
+-- 30 (bf=+0)
|       +-- 25 (bf=+0)
|   +-- 20 (bf=+0)
|   |   +-- 10 (bf=+0)

Level-order balance factors:
  L0: 30(bf=+0)
  L1: 20(bf=+0), 40(bf=+0)
  L2: 10(bf=+0), 25(bf=+0), 35(bf=+0), 50(bf=+0)

PERFECTLY balanced! Every node has bf=0.
AVL tree height = 3 (optimal for 7 nodes).


---
## Final AVL Tree Summary

In [9]:
print("=" * 55)
print("FINAL AVL TREE (insert: [30, 20, 10, 25, 40, 35, 50])")
print("=" * 55)
avl.print_tree()
print()
avl.print_bfs()
print()
print(f"Tree height: {avl.get_height(avl.root)}")
print(f"Node count:  {len(avl.inorder())}")
print()
print("ASCII structure:")
print(r'''
          30 (bf=0)
         /  \
    20(bf=0)  40(bf=0)
    /  \      /  \
 10    25  35    50
(bf=0) (bf=0) (bf=0) (bf=0)
''')
print("Perfectly balanced AVL tree -- all bf=0!")

FINAL AVL TREE (insert: [30, 20, 10, 25, 40, 35, 50])
    |   +-- 50 (bf=+0)
    +-- 40 (bf=+0)
        +-- 35 (bf=+0)
+-- 30 (bf=+0)
|       +-- 25 (bf=+0)
|   +-- 20 (bf=+0)
|   |   +-- 10 (bf=+0)

Level-order balance factors:
  L0: 30(bf=+0)
  L1: 20(bf=+0), 40(bf=+0)
  L2: 10(bf=+0), 25(bf=+0), 35(bf=+0), 50(bf=+0)

Tree height: 3
Node count:  7

ASCII structure:

          30 (bf=0)
         /  \
    20(bf=0)  40(bf=0)
    /  \      /  \
 10    25  35    50
(bf=0) (bf=0) (bf=0) (bf=0)

Perfectly balanced AVL tree -- all bf=0!


---
## Inorder Traversal -- Verify BST Property

In [10]:
result = avl.inorder()
print("Inorder traversal:", result)

is_sorted = all(result[i] < result[i+1] for i in range(len(result)-1))
expected = sorted([30, 20, 10, 25, 40, 35, 50])

print(f"Strictly increasing: {is_sorted}")
print(f"Expected: {expected}")
print(f"Actual:   {result}")
print(f"Match:    {result == expected}")
print()
if result == expected and is_sorted:
    print("BST property VERIFIED -- inorder gives sorted ascending order!")

Inorder traversal: [10, 20, 25, 30, 35, 40, 50]
Strictly increasing: True
Expected: [10, 20, 25, 30, 35, 40, 50]
Actual:   [10, 20, 25, 30, 35, 40, 50]
Match:    True

BST property VERIFIED -- inorder gives sorted ascending order!


---
## Rotation Summary

| Step | Insert | Imbalanced Node | Its BF | Child BF | Type | Rotation | Result |
|:----:|:------:|:--------------:|:------:|:--------:|:----:|----------|--------|
| 3 | 10 | 30 | +2 | +1 (20) | **LL** | Right(30) | 20 becomes root, height 3->2 |
| 6 | 35 | 20 | -2 | -1 (30) | **RR** | Left(20) | 30 becomes root, tree rebalanced |
| 7 | 50 | -- | -- | -- | **None** | -- | Tree becomes perfectly balanced! |

### Rotation Type Distribution

This sequence demonstrates **LL** and **RR** (the two single rotations). LR and RL were not triggered.

| Rotation | Occurrences | Trigger Condition |
|----------|:-----------:|-------------------|
| LL | 1 | Node BF=+2, Left child BF>=0 |
| RR | 1 | Node BF=-2, Right child BF<=0 |
| LR | 0 | Node BF=+2, Left child BF<0 |
| RL | 0 | Node BF=-2, Right child BF>0 |

### Key Takeaways

1. **Only 2 rotations for 7 insertions**: AVL insertion overhead is small in practice
2. **Use child BF, not inserted value**: The correct way to determine rotation type
3. **Bottom-up repair**: Balance factors are checked along the entire insertion path
4. **Final tree height = 3**: Plain BST worst-case height = 7 (linked list). AVL: optimal O(log N)
5. **Step 7 improved balance**: Inserting 50 filled 40's gap, making the tree perfectly balanced (all bf=0)

---
## AVL vs Plain BST

| Metric | Plain BST (worst) | AVL Tree (this example) |
|--------|:-----------------:|:-----------------------:|
| Height after 7 inserts | 5-7 | **3** |
| Search complexity | O(N) | **O(log N)** |
| Total rotations | 0 | **2** (both single) |
| Final tree shape | Degenerate chain | **Perfectly balanced** (all bf=0) |
| Insert overhead | O(h) | O(h) + O(1) rotation |

### Plain BST (no balancing) would look like:

```
30
  \
   20
     \
      10
        \
         25
           \
            40
           /
         35
           \
            50
```
Height = 5, search worst case = 5 comparisons.

### AVL Tree (balanced):

```
          30
         /  \
       20    40
      / \   / \
    10  25 35  50
```
Height = 3, search worst case = 3 comparisons. **40% faster** for just 7 nodes; the difference grows exponentially with N.